# Claude용 Custom Skills 구축

조직의 특화 지식과 workflow로 Claude의 능력을 확장하는 custom skill을 생성, 배포, 관리하는 방법을 배웁니다.

## 목차

1. [소개 및 설정](#introduction)
2. [Custom Skills Architecture 이해](#architecture)
3. [예제 1: 재무 비율 계산기](#financial-ratio)
4. [예제 2: 회사 Brand 지침](#brand-guidelines)
5. [예제 3: 재무 Modeling Suite](#financial-modeling)
6. [Skill 관리 및 Versioning](#management)
7. [모범 사례 및 Production 팁](#best-practices)
8. [문제 해결](#troubleshooting)

## 1. 소개 및 설정 {#introduction}

### Custom Skills란 무엇인가요?

**Custom skill**은 Claude에게 조직 고유의 workflow, 도메인 지식, 모범 사례를 가르치기 위해 만드는 특화 전문성 package입니다. Anthropic의 사전 제작 skill(Excel, PowerPoint, PDF)과 달리 custom skill은 다음을 가능하게 합니다.

- **조직 지식의 규칙화** - 팀의 구체적인 방법론을 담기
- **일관성 보장** - 모든 상호작용에 동일한 기준 적용
- **복잡한 workflow 자동화** - 여러 단계의 과정을 연결
- **지적 재산 유지** - 독점적인 방법을 안전하게 유지

### 핵심 이점

| 이점 | 설명 |
|---------|-------------|
| **전문성 확장** | 모든 Claude 상호작용에 특화 지식 배포 |
| **Version Control** | 변경 추적 및 필요 시 rollback |
| **조합 가능성** | 복잡한 작업을 위해 여러 skill 결합 |
| **Privacy** | skill은 조직에 비공개로 유지 |

### 사전 요구 사항

시작 전에 다음을 갖추었는지 확인하세요.
- [Notebook 1: Skills 소개](01_skills_introduction.ipynb) 완료
- Skills beta 접근 권한이 있는 Anthropic API key
- local SDK가 설치된 Python 환경

### 환경 설정

환경을 설정하고 필요한 library를 import합니다.

In [ ]:
import os
import sys
from pathlib import Path
from typing import Any

# import를 위해 상위 directory 추가
sys.path.insert(0, str(Path.cwd().parent))

from anthropic import Anthropic
from anthropic.lib import files_from_dir
from dotenv import load_dotenv

# 우리의 utility import
from file_utils import (
    download_all_files,
    extract_file_ids,
    print_download_summary,
)

# 이 notebook에서 나중에 skill_utils 생성
# from skill_utils import (
#     create_skill,
#     list_skills,
#     delete_skill,
#     test_skill
# )

# 환경 변수 로딩
load_dotenv(Path.cwd().parent / ".env")

API_KEY = os.getenv("ANTHROPIC_API_KEY")
MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6")

if not API_KEY:
    raise ValueError(
        "ANTHROPIC_API_KEY not found. Copy ../.env.example to ../.env and add your API key."
    )

# Skills beta로 client 초기화
client = Anthropic(api_key=API_KEY, default_headers={"anthropic-beta": "skills-2025-10-02"})

# directory 설정
SKILLS_DIR = Path.cwd().parent / "custom_skills"
OUTPUT_DIR = Path.cwd().parent / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("✓ API key loaded")
print(f"✓ Using model: {MODEL}")
print(f"✓ Custom skills directory: {SKILLS_DIR}")
print(f"✓ Output directory: {OUTPUT_DIR}")
print("\n📝 Skills beta header configured for skill management")

## 2. Custom Skills Architecture 이해 {#architecture}

### Skill 구조

모든 custom skill은 다음 directory 구조를 따릅니다.

```
skill_name/
├── SKILL.md           # 필수: YAML frontmatter가 있는 지시
├── *.md               # 선택: 추가 .md 파일(문서, 가이드)
├── scripts/           # 선택: 실행 코드
│   ├── process.py
│   └── utils.js
└── resources/         # 선택: template, data 파일
    └── template.xlsx
```

**중요:** 
- **SKILL.md가 유일한 필수 파일** - 그 밖의 모든 것은 선택 사항입니다.
- **여러 .md 파일 허용** - 최상위 folder에 markdown 파일을 원하는 수만큼 둘 수 있습니다.
- **모든 .md 파일 로드** - SKILL.md와 REFERENCE.md뿐 아니라 포함한 모든 .md 파일입니다.
- **필요에 맞게 구성** - 여러 .md 파일로 복잡한 문서를 구성하세요.

📖 engineering 블로그 [Skills로 agent가 실제 세계에 대응할 수 있게 하기](https://www.anthropic.com/engineering/equipping-agents-for-the-real-world-with-agent-skills)를 읽어 보세요.

### Skills는 Markdown만이 아닙니다

![Skills에 Script와 파일 포함 가능](../assets/not-just-markdown.png)

Skills는 다양한 파일 유형을 묶을 수 있습니다.
- **Markdown 파일**: 지시와 문서(SKILL.md, REFERENCE.md 등)
- **Script**: 복잡한 작업용 Python, JavaScript 또는 다른 실행 코드
- **Template**: 사용자 지정 가능한 사전 제작 파일(Excel template, 문서 template)
- **Resource**: 지원 data 파일, 설정, asset

### SKILL.md 요구 사항

`SKILL.md` 파일에는 다음이 포함되어야 합니다.

1. **YAML Frontmatter**(name: 64자, description: 1024자)
   - `name`: 소문자 영숫자와 hyphen(필수)
   - `description`: skill이 하는 일의 간단한 설명(필수)

2. **지시**(markdown 형식)
   - Claude를 위한 명확한 안내
   - 사용 예제
   - 모든 제약과 규칙
   - 권장: 5,000 token 미만 유지

### 추가 문서 파일

더 잘 구성하도록 여러 markdown 파일을 포함할 수 있습니다.

```
skill_name/
├── SKILL.md           # 주요 지시(필수)
├── REFERENCE.md       # API reference(선택)
├── EXAMPLES.md        # 사용 예제(선택)
├── TROUBLESHOOTING.md # 일반적인 문제(선택)
└── CHANGELOG.md       # 버전 이력(선택)
```

skill이 로드되면 root directory의 모든 `.md` 파일을 Claude가 사용할 수 있습니다.

### 묶인 파일 예제

![Skills의 묶인 파일](../assets/skills-bundled-files.png)

이 예제는 Skills가 여러 파일을 묶는 방법을 보여 줍니다.
- **SKILL.md**: 색상, typography, section이 포함된 주요 지시
- **slide-decks.md**: 특정 사용 사례의 추가 문서
- **Script와 resource**: skill 실행 중 참조하고 사용 가능

### Progressive Disclosure

Skills는 token 사용을 최적화하도록 세 단계로 로드됩니다.

| 단계 | 내용 | Token 비용 | 로드 시점 |
|-------|---------|------------|-------------|
| **1. Metadata** | 이름 및 설명 | name: 64자, description: 1024자 | 항상 표시 |
| **2. 지시** | 모든 .md 파일 | 5,000 token 미만 권장 | 관련될 때 |
| **3. Resource** | Script 및 파일 | 필요에 따라 | 실행 중 |

### API Workflow

```python
# 1. skill 생성
skill = client.beta.skills.create(
    display_title="My Skill",
    files=files_from_dir("path/to/skill")
)

# 2. message에서 사용
response = client.beta.messages.create(
    container={
        "skills": [{
            "type": "custom",
            "skill_id": skill.id,
            "version": "latest"
        }]
    },
    # ... 나머지 message parameter
)
```

### 모범 사례

skill 생성과 모범 사례의 자세한 지침:
- [Claude Skills 모범 사례](https://docs.claude.com/en/docs/agents-and-tools/agent-skills/best-practices)
- [Skills 문서](https://docs.claude.com/en/docs/agents-and-tools/agent-skills/overview)

### Skill Utility 함수 생성

skill 관리를 위한 helper 함수를 만듭니다.

In [ ]:
def create_skill(client: Anthropic, skill_path: str, display_title: str) -> dict[str, Any]:
    """
    directory에서 새 custom skill을 생성합니다.

    Args:
        client: Anthropic client instance
        skill_path: skill directory 경로
        display_title: 사람이 읽을 수 있는 skill 이름

    Returns:
        skill_id, version, metadata가 있는 dictionary
    """
    try:
        # files_from_dir로 skill 생성
        skill = client.beta.skills.create(
            display_title=display_title, files=files_from_dir(skill_path)
        )

        return {
            "success": True,
            "skill_id": skill.id,
            "display_title": skill.display_title,
            "latest_version": skill.latest_version,
            "created_at": skill.created_at,
            "source": skill.source,
        }
    except Exception as e:
        return {"success": False, "error": str(e)}


def list_custom_skills(client: Anthropic) -> list[dict[str, Any]]:
    """
    workspace의 모든 custom skill을 나열합니다.

    Returns:
        skill dictionary 목록
    """
    try:
        skills_response = client.beta.skills.list(source="custom")

        skills = []
        for skill in skills_response.data:
            skills.append(
                {
                    "skill_id": skill.id,
                    "display_title": skill.display_title,
                    "latest_version": skill.latest_version,
                    "created_at": skill.created_at,
                    "updated_at": skill.updated_at,
                }
            )

        return skills
    except Exception as e:
        print(f"Error listing skills: {e}")
        return []


def delete_skill(client: Anthropic, skill_id: str) -> bool:
    """
    custom skill과 모든 버전을 삭제합니다.

    Args:
        client: Anthropic client
        skill_id: 삭제할 skill ID

    Returns:
        성공하면 True, 아니면 False
    """
    try:
        # 먼저 모든 버전 삭제
        versions = client.beta.skills.versions.list(skill_id=skill_id)

        for version in versions.data:
            client.beta.skills.versions.delete(skill_id=skill_id, version=version.version)

        # 그다음 skill 자체 삭제
        client.beta.skills.delete(skill_id)
        return True

    except Exception as e:
        print(f"Error deleting skill: {e}")
        return False


def test_skill(
    client: Anthropic,
    skill_id: str,
    test_prompt: str,
    model: str = "claude-sonnet-4-6",
) -> Any:
    """
    프롬프트로 custom skill을 테스트합니다.

    Args:
        client: Anthropic client
        skill_id: 테스트할 skill ID
        test_prompt: skill 테스트 프롬프트
        model: 테스트에 사용할 model

    Returns:
        Claude의 응답
    """
    response = client.beta.messages.create(
        model=model,
        max_tokens=4096,
        container={"skills": [{"type": "custom", "skill_id": skill_id, "version": "latest"}]},
        tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
        messages=[{"role": "user", "content": test_prompt}],
        betas=[
            "code-execution-2025-08-25",
            "files-api-2025-04-14",
            "skills-2025-10-02",
        ],
    )

    return response


print("✓ Skill utility functions defined")
print("  - create_skill()")
print("  - list_custom_skills()")
print("  - delete_skill()")
print("  - test_skill()")

### 기존 Custom Skills 확인

workspace에 custom skill이 이미 있는지 확인합니다.

### ⚠️ 중요: 시작 전 기존 Skills 정리

notebook을 다시 실행한다면 이전 session의 skill이 있을 수 있습니다. Skills는 display title이 중복될 수 없으므로 선택지는 세 가지입니다.

1. **기존 skill 삭제**(테스트에 권장) - 깨끗한 상태에서 시작
2. **다른 display title 사용** - 이름에 timestamp나 버전 번호 추가
3. **새 버전으로 기존 skill 갱신** - [Skill 관리 및 Versioning](#management) 섹션 참조

기존 skill을 확인하고 선택적으로 정리합니다.

In [ ]:
# 충돌할 수 있는 기존 skill 확인
existing_skills = list_custom_skills(client)
skill_titles_to_create = [
    "Financial Ratio Analyzer",
    "Corporate Brand Guidelines",
    "Financial Modeling Suite",
]
conflicting_skills = []

if existing_skills:
    print(f"Found {len(existing_skills)} existing custom skill(s):")
    for skill in existing_skills:
        print(f"  - {skill['display_title']} (ID: {skill['skill_id']})")
        if skill["display_title"] in skill_titles_to_create:
            conflicting_skills.append(skill)

    if conflicting_skills:
        print(
            f"\n⚠️ Found {len(conflicting_skills)} skill(s) that will conflict with this notebook:"
        )
        for skill in conflicting_skills:
            print(f"  - {skill['display_title']} (ID: {skill['skill_id']})")

        print("\n" + "=" * 70)
        print("To clean up these skills and start fresh, uncomment and run:")
        print("=" * 70)
        print("\n# UNCOMMENT THE LINES BELOW TO DELETE CONFLICTING SKILLS:")
        print("# for skill in conflicting_skills:")
        print("#     if delete_skill(client, skill['skill_id']):")
        print("#         print(f\"✅ Deleted: {skill['display_title']}\")")
        print("#     else:")
        print("#         print(f\"❌ Failed to delete: {skill['display_title']}\")")

        # for skill in conflicting_skills:
        #     if delete_skill(client, skill['skill_id']):
        #         print(f"✅ Deleted: {skill['display_title']}")
        #     else:
        #         print(f"❌ Failed to delete: {skill['display_title']}")
    else:
        print("\n✅ No conflicting skills found. Ready to proceed!")
else:
    print("✅ No existing custom skills found. Ready to create new ones!")

## 3. 예제 1: 재무 비율 계산기 {#financial-ratio}

회사 재무 건전성을 분석할 수 있는 재무 비율 계산기를 첫 custom skill로 만듭니다.

### Skill 개요

**재무 비율 계산기** skill은:
- 핵심 재무 비율 계산(ROE, P/E, Current Ratio 등)
- 업계 맥락으로 비율 해석
- 서식 지정 보고서 생성
- 다양한 data 형식 사용(CSV, JSON, 텍스트)

### Financial Analyzer Skill 업로드

이제 financial analyzer skill을 Claude에 업로드합니다.

In [ ]:
# Financial Analyzer skill 업로드
financial_skill_path = SKILLS_DIR / "analyzing-financial-statements"

if financial_skill_path.exists():
    print("Uploading Financial Analyzer skill...")
    result = create_skill(client, str(financial_skill_path), "Financial Ratio Analyzer")

    if result["success"]:
        financial_skill_id = result["skill_id"]
        print("✅ Skill uploaded successfully!")
        print(f"   Skill ID: {financial_skill_id}")
        print(f"   Version: {result['latest_version']}")
        print(f"   Created: {result['created_at']}")
    else:
        print(f"❌ Upload failed: {result['error']}")
        if "cannot reuse an existing display_title" in str(result["error"]):
            print("\n💡 Solution: A skill with this name already exists.")
            print("   Run the 'Clean Up Existing Skills' cell above to delete it first,")
            print("   or change the display_title to something unique.")
else:
    print(f"⚠️ Skill directory not found: {financial_skill_path}")
    print(
        "Please ensure the custom_skills directory contains the analyzing-financial-statements folder."
    )

### Financial Analyzer Skill 테스트

sample 재무 data로 skill을 테스트합니다.

In [ ]:
# Financial Analyzer skill 테스트
if "financial_skill_id" in locals():
    test_prompt = """
    Calculate financial ratios for this company:

    Income Statement:
    - Revenue: $1,000M
    - EBITDA: $200M
    - Net Income: $120M

    Balance Sheet:
    - Total Assets: $2,000M
    - Current Assets: $500M
    - Current Liabilities: $300M
    - Total Debt: $400M
    - Shareholders Equity: $1,200M

    Market Data:
    - Share Price: $50
    - Shares Outstanding: 100M

    Please calculate key ratios and provide analysis.
    """

    print("Testing Financial Analyzer skill...")
    response = test_skill(client, financial_skill_id, test_prompt)

    # 응답 출력
    for content in response.content:
        if content.type == "text":
            print(content.text)
else:
    print("⚠️ Please upload the Financial Analyzer skill first (run the previous cell)")

## 4. 예제 2: 회사 Brand 지침 {#brand-guidelines}

이제 모든 문서가 회사 brand 기준을 따르게 하는 skill을 만듭니다.

### Skill 개요

**Brand 지침** skill은:
- 일관된 색상, font, layout 적용
- logo 배치와 사용 보장
- 전문적인 어조와 메시지 유지
- 모든 문서 유형에서 작동(Excel, PowerPoint, PDF)

In [ ]:
# Brand Guidelines skill 업로드
brand_skill_path = SKILLS_DIR / "applying-brand-guidelines"

if brand_skill_path.exists():
    print("Uploading Brand Guidelines skill...")
    result = create_skill(client, str(brand_skill_path), "Corporate Brand Guidelines")

    if result["success"]:
        brand_skill_id = result["skill_id"]
        print("✅ Skill uploaded successfully!")
        print(f"   Skill ID: {brand_skill_id}")
        print(f"   Version: {result['latest_version']}")
    else:
        print(f"❌ Upload failed: {result['error']}")
        if "cannot reuse an existing display_title" in str(result["error"]):
            print("\n💡 Solution: A skill with this name already exists.")
            print("   Run the 'Clean Up Existing Skills' cell above to delete it first,")
            print("   or change the display_title to something unique.")
else:
    print(f"⚠️ Skill directory not found: {brand_skill_path}")

### 문서 생성으로 Brand 지침 테스트

brand를 적용한 PowerPoint presentation을 만들어 brand skill을 테스트합니다.

In [ ]:
# PowerPoint 생성으로 Brand Guidelines skill 테스트
if "brand_skill_id" in locals():
    # brand skill과 Anthropic의 pptx skill 결합
    response = client.beta.messages.create(
        model=MODEL,
        max_tokens=4096,
        container={
            "skills": [
                {"type": "custom", "skill_id": brand_skill_id, "version": "latest"},
                {"type": "anthropic", "skill_id": "pptx", "version": "latest"},
            ]
        },
        tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
        messages=[
            {
                "role": "user",
                "content": """Create a 3-slide PowerPoint presentation following Acme Corporation brand guidelines:

            Slide 1: Title slide for "Q4 2025 Results"
            Slide 2: Revenue Overview with a chart showing Q1-Q4 growth
            Slide 3: Key Achievements (3 bullet points)

            Apply all brand colors, fonts, and formatting standards.
            """,
            }
        ],
        betas=[
            "code-execution-2025-08-25",
            "files-api-2025-04-14",
            "skills-2025-10-02",
        ],
    )

    print("Response from Claude:")
    for content in response.content:
        if content.type == "text":
            print(content.text[:500] + "..." if len(content.text) > 500 else content.text)

    # 생성 파일 다운로드
    file_ids = extract_file_ids(response)
    if file_ids:
        results = download_all_files(
            client, response, output_dir=str(OUTPUT_DIR), prefix="branded_"
        )
        print_download_summary(results)
else:
    print("⚠️ Please upload the Brand Guidelines skill first")

## 5. 예제 3: 재무 Modeling Suite {#financial-modeling}

가장 고급 skill인 가치 평가와 risk 분석용 종합 재무 modeling suite를 만듭니다.

### Skill 개요

**재무 Modeling Suite** skill이 제공하는 것:
- **DCF Valuation**: 완전한 discounted cash flow model
- **Sensitivity Analysis**: 변수의 valuation 영향 테스트
- **Monte Carlo Simulation**: 확률분포를 사용한 risk modeling
- **Scenario Planning**: 최선/기본/최악 사례 분석

복잡한 계산과 전문적인 수준의 재무 modeling을 갖춘 multi-file skill을 보여 줍니다.

### 재무 Modeling Suite 업로드 

먼저 재무 modeling skill을 업로드합니다.

In [ ]:
# Financial Modeling Suite skill 업로드
modeling_skill_path = SKILLS_DIR / "creating-financial-models"

if modeling_skill_path.exists():
    print("Uploading Financial Modeling Suite skill...")
    result = create_skill(client, str(modeling_skill_path), "Financial Modeling Suite")

    if result["success"]:
        modeling_skill_id = result["skill_id"]
        print("✅ Skill uploaded successfully!")
        print(f"   Skill ID: {modeling_skill_id}")
        print(f"   Version: {result['latest_version']}")
        print("\nThis skill includes:")
        print("   - DCF valuation model (dcf_model.py)")
        print("   - Sensitivity analysis framework (sensitivity_analysis.py)")
        print("   - Monte Carlo simulation capabilities")
        print("   - Scenario planning tools")
    else:
        print(f"❌ Upload failed: {result['error']}")
else:
    print(f"⚠️ Skill directory not found: {modeling_skill_path}")
    print(
        "Please ensure the custom_skills directory contains the creating-financial-models folder."
    )

### 재무 Modeling Suite 테스트

DCF valuation 요청으로 고급 modeling 능력을 테스트합니다.

In [ ]:
# DCF valuation으로 Financial Modeling Suite 테스트
if "modeling_skill_id" in locals():
    dcf_test_prompt = """
    Perform a DCF valuation for TechCorp with the following data:

    Historical Financials (Last 3 Years):
    - Revenue: $500M, $600M, $750M
    - EBITDA Margin: 25%, 27%, 30%
    - CapEx: $50M, $55M, $60M
    - Working Capital: 15% of revenue

    Projections:
    - Revenue growth: 20% for years 1-3, then declining to 5% by year 5
    - EBITDA margin expanding to 35% by year 5
    - Terminal growth rate: 3%

    Market Assumptions:
    - WACC: 10%
    - Tax rate: 25%
    - Current net debt: $200M
    - Shares outstanding: 100M

    Please create a complete DCF model with sensitivity analysis on WACC and terminal growth.
    Generate an Excel file with the full model including:
    1. Revenue projections
    2. Free cash flow calculations
    3. Terminal value
    4. Enterprise value to equity value bridge
    5. Sensitivity table
    """

    print("Testing Financial Modeling Suite with DCF valuation...")
    print("=" * 70)
    print("\n⏱️ Note: Complex financial model generation may take 1-2 minutes.\n")

    response = client.beta.messages.create(
        model=MODEL,
        max_tokens=4096,
        container={
            "skills": [
                {"type": "custom", "skill_id": modeling_skill_id, "version": "latest"},
                {"type": "anthropic", "skill_id": "xlsx", "version": "latest"},
            ]
        },
        tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
        messages=[{"role": "user", "content": dcf_test_prompt}],
        betas=[
            "code-execution-2025-08-25",
            "files-api-2025-04-14",
            "skills-2025-10-02",
        ],
    )

    # Claude 응답 출력
    for content in response.content:
        if content.type == "text":
            # 출력을 적절히 유지하도록 첫 800자 출력
            text = content.text
            if len(text) > 800:
                print(text[:800] + "\n\n[... Output truncated for brevity ...]")
            else:
                print(text)

    # 생성됐으면 DCF model 다운로드
    file_ids = extract_file_ids(response)
    if file_ids:
        print("\n" + "=" * 70)
        print("Downloading generated DCF model...")
        results = download_all_files(
            client, response, output_dir=str(OUTPUT_DIR), prefix="dcf_model_"
        )
        print_download_summary(results)
        print("\n💡 Open the Excel file to explore the complete DCF valuation model!")
else:
    print("⚠️ Please upload the Financial Modeling Suite skill first (run the previous cell)")

## 6. Skill 관리 및 Versioning {#management}

시간이 지나며 skill을 관리하려면 versioning, 갱신, lifecycle 관리를 이해해야 합니다.

### Skills 나열

workspace의 모든 custom skill 개요를 얻습니다.

In [ ]:
# 자신의 모든 custom skill 나열
my_skills = list_custom_skills(client)

if my_skills:
    print(f"You have {len(my_skills)} custom skill(s):\n")
    print("=" * 70)
    for i, skill in enumerate(my_skills, 1):
        print(f"\n{i}. {skill['display_title']}")
        print(f"   Skill ID: {skill['skill_id']}")
        print(f"   Current Version: {skill['latest_version']}")
        print(f"   Created: {skill['created_at']}")
        if skill.get("updated_at"):
            print(f"   Last Updated: {skill['updated_at']}")
    print("\n" + "=" * 70)
else:
    print("No custom skills found in your workspace.")

### 새 버전 생성

Skills는 history 유지와 rollback을 위한 versioning을 지원합니다. Financial Analyzer skill을 개선하고 새 버전을 만듭니다.

#### 1단계: Financial Analyzer 개선

skill을 더 다양하게 사용할 수 있게 **의료 업계** benchmark를 추가합니다. 사용자 요구에 따라 skill 능력을 확장하는 실제 scenario입니다.

In [ ]:
# Financial Analyzer에 의료 업계 benchmark 추가
# 실제적인 skill 개선 scenario 시연

if "financial_skill_id" in locals():
    # 현재 interpret_ratios.py 파일 읽기
    interpret_file_path = SKILLS_DIR / "analyzing-financial-statements" / "interpret_ratios.py"

    with open(interpret_file_path) as f:
        content = f.read()

    # 'manufacturing' section 뒤에 의료 benchmark 추가
    healthcare_benchmarks = """        },
        'healthcare': {
            'current_ratio': {'excellent': 2.3, 'good': 1.8, 'acceptable': 1.4, 'poor': 1.0},
            'debt_to_equity': {'excellent': 0.3, 'good': 0.6, 'acceptable': 1.0, 'poor': 1.8},
            'roe': {'excellent': 0.22, 'good': 0.16, 'acceptable': 0.11, 'poor': 0.07},
            'gross_margin': {'excellent': 0.65, 'good': 0.45, 'acceptable': 0.30, 'poor': 0.20},
            'pe_ratio': {'undervalued': 18, 'fair': 28, 'growth': 40, 'expensive': 55}
        """

    # manufacturing section 뒤, 닫는 중괄호 앞 위치 찾기
    insert_pos = content.find("        }\n    }")  # BENCHMARKS dict 끝 찾기

    if insert_pos != -1:
        # 의료 benchmark 삽입
        new_content = content[:insert_pos] + healthcare_benchmarks + content[insert_pos:]

        # 개선한 파일 저장
        with open(interpret_file_path, "w") as f:
            f.write(new_content)

        print("✅ Enhanced Financial Analyzer with healthcare industry benchmarks")
        print("\nChanges made:")
        print("  - Added healthcare industry to BENCHMARKS")
        print("  - Includes specific thresholds for:")
        print("    • Current ratio (liquidity)")
        print("    • Debt-to-equity (leverage)")
        print("    • ROE (profitability)")
        print("    • Gross margin")
        print("    • P/E ratio (valuation)")
        print("\n📝 Now we can create a new version of the skill with this enhancement!")
    else:
        print("⚠️ Could not find the correct position to insert healthcare benchmarks")
        print("The file structure may have changed.")
else:
    print("⚠️ Please upload the Financial Analyzer skill first (run cells in Section 3)")

#### 2단계: 새 버전 생성

skill을 개선했으니 변경을 추적할 새 버전을 만듭니다.

In [ ]:
# 개선한 Financial Analyzer skill 새 버전 생성
def create_skill_version(client: Anthropic, skill_id: str, skill_path: str):
    """기존 skill의 새 버전을 생성합니다."""
    try:
        version = client.beta.skills.versions.create(
            skill_id=skill_id, files=files_from_dir(skill_path)
        )
        return {
            "success": True,
            "version": version.version,
            "created_at": version.created_at,
        }
    except Exception as e:
        return {"success": False, "error": str(e)}


# 의료 개선 사항으로 새 버전 생성
if "financial_skill_id" in locals():
    print("Creating new version of Financial Analyzer with healthcare benchmarks...")

    result = create_skill_version(
        client, financial_skill_id, str(SKILLS_DIR / "analyzing-financial-statements")
    )

    if result["success"]:
        print("✅ New version created successfully!")
        print(f"   Version: {result['version']}")
        print(f"   Created: {result['created_at']}")
        print("\n📊 Version History:")
        print("   v1: Original skill with tech, retail, financial, manufacturing")
        print(f"   v{result['version']}: Enhanced with healthcare industry benchmarks")
    else:
        print(f"❌ Version creation failed: {result['error']}")
else:
    print("⚠️ Please run the previous cells to upload the skill and make enhancements first")

#### 3단계: 새 버전 테스트

의료 회사를 분석하여 개선 사항이 작동하는지 확인합니다.

In [ ]:
# 의료 업계 data로 개선한 skill 테스트
if "financial_skill_id" in locals():
    healthcare_test_prompt = """
    Analyze this healthcare company using the healthcare industry benchmarks:

    Company: MedTech Solutions (Healthcare Industry)

    Income Statement:
    - Revenue: $800M
    - EBITDA: $320M
    - Net Income: $160M

    Balance Sheet:
    - Total Assets: $1,200M
    - Current Assets: $400M
    - Current Liabilities: $200M
    - Total Debt: $300M
    - Shareholders Equity: $700M

    Market Data:
    - Share Price: $75
    - Shares Outstanding: 50M

    Please calculate key ratios and provide healthcare-specific analysis.
    """

    print("Testing enhanced Financial Analyzer with healthcare company...")
    print("=" * 70)

    response = test_skill(client, financial_skill_id, healthcare_test_prompt, MODEL)

    # Claude 분석 출력
    for content in response.content:
        if content.type == "text":
            # 출력을 적절히 유지하도록 첫 1000자 출력
            text = content.text
            if len(text) > 1000:
                print(text[:1000] + "\n\n[... Output truncated for brevity ...]")
            else:
                print(text)

    print(
        "\n✅ The skill now recognizes 'healthcare' as an industry and applies specific benchmarks!"
    )
else:
    print("⚠️ Please run the previous cells to create the enhanced version first")

### 정리: Skills 관리

테스트를 마쳤거나 workspace 정리가 필요하면 skill을 선택적으로 제거할 수 있습니다. 생성한 것을 검토하고 정리 option을 제공합니다.

In [ ]:
# 상세 보고를 포함한 종합 skill 정리
def review_and_cleanup_skills(client, dry_run=True):
    """
    모든 skill을 검토하고 선택적으로 이 notebook에서 만든 것을 정리합니다.

    Args:
        client: Anthropic client
        dry_run: True이면 실제로 삭제하지 않고 삭제할 것만 표시
    """
    # 현재 모든 skill 얻기
    all_skills = list_custom_skills(client)

    # 이 notebook에서 만든 skill
    notebook_skill_names = [
        "Financial Ratio Analyzer",
        "Corporate Brand Guidelines",
        "Financial Modeling Suite",
    ]

    # 이 notebook에서 만든 skill 추적
    notebook_skills = []
    other_skills = []

    for skill in all_skills:
        if skill["display_title"] in notebook_skill_names:
            notebook_skills.append(skill)
        else:
            other_skills.append(skill)

    print("=" * 70)
    print("SKILL INVENTORY REPORT")
    print("=" * 70)

    print(f"\nTotal custom skills in workspace: {len(all_skills)}")

    if notebook_skills:
        print(f"\n📚 Skills created by this notebook ({len(notebook_skills)}):")
        for skill in notebook_skills:
            print(f"   • {skill['display_title']}")
            print(f"     ID: {skill['skill_id']}")
            print(f"     Version: {skill['latest_version']}")
            print(f"     Created: {skill['created_at']}")
    else:
        print("\n✅ No skills from this notebook found")

    if other_skills:
        print(f"\n🔧 Other skills in workspace ({len(other_skills)}):")
        for skill in other_skills:
            print(f"   • {skill['display_title']} (v{skill['latest_version']})")

    # 정리 option
    if notebook_skills:
        print("\n" + "=" * 70)
        print("CLEANUP OPTIONS")
        print("=" * 70)

        if dry_run:
            print("\n🔍 DRY RUN MODE - No skills will be deleted")
            print("\nTo delete the notebook skills, uncomment and run:")
            print("-" * 40)
            print("# review_and_cleanup_skills(client, dry_run=False)")
            print("-" * 40)

            print("\nThis would delete:")
            for skill in notebook_skills:
                print(f"   • {skill['display_title']}")
        else:
            print("\n⚠️ DELETION MODE - Skills will be permanently removed")
            print("\nDeleting notebook skills...")

            success_count = 0
            for skill in notebook_skills:
                if delete_skill(client, skill["skill_id"]):
                    print(f"   ✅ Deleted: {skill['display_title']}")
                    success_count += 1
                else:
                    print(f"   ❌ Failed to delete: {skill['display_title']}")

            print(f"\n📊 Cleanup complete: {success_count}/{len(notebook_skills)} skills deleted")

    return {
        "total_skills": len(all_skills),
        "notebook_skills": len(notebook_skills),
        "other_skills": len(other_skills),
        "notebook_skill_ids": [s["skill_id"] for s in notebook_skills],
    }


# 검토 실행(기본은 dry-run mode)
print("Reviewing your custom skills workspace...")
cleanup_summary = review_and_cleanup_skills(client, dry_run=True)

# 나중 정리를 위해 skill ID 저장
if cleanup_summary["notebook_skill_ids"]:
    skills_to_cleanup = cleanup_summary["notebook_skill_ids"]
    print(f"\n💡 Tip: {len(skills_to_cleanup)} skill(s) can be cleaned up when you're done testing")

# NOTEBOOK SKILL을 실제 삭제하려면 아래 줄 주석 해제:
# review_and_cleanup_skills(client, dry_run=False)

## 7. 모범 사례 및 Production 팁 {#best-practices}

### Skill 설계 원칙

1. **단일 책임**: 각 skill은 하나의 전문성 영역에 집중
2. **명확한 문서**: SKILL.md는 포괄적이면서 간결해야 함
3. **Error Handling**: script가 edge case를 적절하게 처리
4. **Version Control**: Git으로 skill 변경 추적
5. **테스트**: production 배포 전 항상 skill 테스트

### Directory 구조 모범 사례

```
custom_skills/
├── financial_analyzer/       # 단일 목적, 명확한 이름
│   ├── SKILL.md             # 5,000 token 미만
│   ├── scripts/             # 모듈화된 Python/JS 파일
│   └── tests/               # script의 unit test
├── brand_guidelines/         # 조직 기준
│   ├── SKILL.md
│   ├── REFERENCE.md         # 추가 문서
│   └── assets/              # logo, template
```

### 성능 최적화

| 전략 | 영향 | 구현 |
|----------|--------|----------------|
| **최소 Frontmatter** | 더 빠른 skill 발견 | name: 64자, description: 1024자 |
| **Lazy Loading** | token 사용 감소 | 필요할 때만 파일 참조 |
| **Skill 조합** | 중복 회피 | 거대한 skill 대신 skill 결합 |
| **Caching** | 더 빠른 응답 | skill container 재사용 |

### 보안 고려 사항

- **API Key**: skill에 credential을 하드코딩하지 않기
- **Data Privacy**: skill 파일에 민감한 data 포함하지 않기
- **Access Control**: Skills는 workspace별임
- **검증**: script의 입력을 정제
- **Audit Trail**: compliance를 위해 skill 사용 기록

## 다음 단계

🎉 **축하합니다!** Claude용 custom skill을 생성, 배포, 관리하는 방법을 배웠습니다.

### 배운 내용

- ✅ Custom skill architecture와 요구 사항
- ✅ SKILL.md와 Python script로 skill 생성
- ✅ API로 skill 업로드
- ✅ custom과 Anthropic skill 결합
- ✅ production 배포 모범 사례
- ✅ 일반적인 문제 해결

### 계속하기

1. **실험**: 사용 사례에 맞게 예제 skill 수정
2. **구축**: 조직의 workflow용 skill 생성
3. **최적화**: token 사용과 성능 모니터링
4. **공유**: 팀 협업을 위해 skill 문서화

### Resource

- [Claude API 문서](https://docs.anthropic.com/en/api/messages)
- [Skills 문서](https://docs.claude.com/en/docs/agents-and-tools/agent-skills/overview)
- [모범 사례](https://docs.claude.com/en/docs/agents-and-tools/agent-skills/best-practices)
- [Files API 문서](https://docs.claude.com/en/api/files-content)
- 예제 Skills Repository (곧 제공)

### 시도할 Skill 아이디어

- 📊 **Data Pipeline**: 검증이 있는 ETL workflow
- 📝 **문서 Template**: 계약서, 제안서, 보고서
- 🔍 **Code Review**: 스타일 가이드와 모범 사례
- 📈 **Analytics Dashboard**: KPI 추적과 시각화
- 🤖 **Automation Suite**: 반복 작업 workflow

즐겁게 skill을 만들어 보세요! 🚀